# KIVI: 2-Bit KV-Cache Quantization for LLaMA (7B & 13B)

**Course:** CMSC 723 — Graduate NLP  
**Author:** Kiyana Amirian  
**Team:** Kiyana Amirian, Helia Mohammadpour, Nick Milionis, Hengyuan Liu, Saketh Challagundla

---

## Overview

This notebook implements and evaluates **KIVI** — a training-free, 2-bit KV-cache quantization scheme for large language models, applied to LLaMA-2 7B and 13B. KIVI significantly reduces KV-cache memory footprint while maintaining generation quality across multiple benchmarks.

### Key idea

Instead of storing full-precision (FP16) key-value tensors in the attention cache, KIVI:
1. Quantizes tokens to **2 bits** (per-channel for Keys, per-token for Values) using group quantization
2. Maintains a small **residual buffer** of recent full-precision tokens to preserve local context
3. Dequantizes on-the-fly during attention computation — no retraining required

### Structure

| Section | Description |
|---------|-------------|
| 1. Quantization primitives | Per-token and per-channel 2-bit quantization with group-size calibration |
| 2. KIVI cache manager | `KIVICache` class: quantized storage + residual buffer + dequantization |
| 3. LLaMA attention replacement | Drop-in KIVI attention module; `replace_llama_attention_with_kivi` |
| 4. Memory profiling | Theoretical KV-cache MB and empirical GPU peak memory |
| 5. CNN/DailyMail evaluation | Summarization benchmark — ROUGE, BERTScore, token match |
| 6. GSM8K evaluation | Math reasoning benchmark — exact match accuracy (7B and 13B) |
| 7. CoQA evaluation | Conversational QA benchmark |

### Results summary

| Model | Benchmark | Baseline | KIVI | Memory reduction |
|-------|-----------|----------|------|------------------|
| LLaMA-2 7B | CNN/DM (ROUGE-L) | — | — | ~8× theoretical |
| LLaMA-2 13B | GSM8K (EM) | — | — | ~8× theoretical |
| LLaMA-2 7B | CoQA | — | — | ~8× theoretical |

> **Note:** Results are stored in `results/`. The model weights require a HuggingFace token with LLaMA-2 access. GPU recommended (A100/H100 for 13B).


## Section 1: Imports and Model Loading

All function/class definitions live in the `kivi/` package and `utils.py`.
Install extra dependencies once with:
```
pip install rouge-score bert-score datasets transformers huggingface_hub
```


In [ ]:
import torch
from transformers import AutoTokenizer, LlamaForCausalLM
from huggingface_hub import login

# --- KIVI package ---
from kivi import (
    KIVICache,
    LlamaAttentionWithKIVI,
    replace_llama_attention_with_kivi,
    reset_all_kivi_caches,
    get_kivi_memory_stats,
    get_model_kv_config,
    measure_baseline_memory,
    generate_baseline_with_output_and_kv,
    get_baseline_kv_memory_stats,
    theoretical_baseline_kv_mb,
    print_per_prompt_kv,
    print_aggregate_kv,
    print_full_run_gpu_memory,
    print_all_memory_results,
    print_kivi_memory_stats,
)
from utils import to_json_safe, token_level_match_rate
import kivi.benchmarks.cnn_dm as cnn_dm_bench
import kivi.benchmarks.coqa as coqa_bench
import kivi.benchmarks.gsm8k as gsm8k_bench


In [ ]:
# Log in to HuggingFace (required for LLaMA-2 gated weights)
# login(token="<YOUR_HF_TOKEN>")

MODEL_7B  = "meta-llama/Llama-2-7b-chat-hf"
MODEL_13B = "meta-llama/Llama-2-13b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(MODEL_7B, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load baseline model (FP16, auto device map)
baseline_model = LlamaForCausalLM.from_pretrained(
    MODEL_7B,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

# Load KIVI model (same weights, attention layers replaced in-place)
kivi_model = LlamaForCausalLM.from_pretrained(
    MODEL_7B,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
kivi_model = replace_llama_attention_with_kivi(kivi_model)

print(f"Baseline parameters: {baseline_model.num_parameters() / 1e9:.2f}B")
print(f"Layers: {len(baseline_model.model.layers)}")


## Section 2: Unit Tests — KIVICache Correctness

Verify that the simple (full-dequant) and split attention implementations agree,
and that the memory statistics API returns sensible numbers.


In [ ]:
# Test 1: simple vs split attention output agreement after prefill
cache = KIVICache(num_bits=2, group_size=32, residual_length=128)

keys   = torch.randn(550, 4096)
values = torch.randn(550, 4096)
cache.prefill(keys, values)

query = torch.randn(1, 4096)

output_simple = cache.compute_attention(query, use_split=False)
output_split  = cache.compute_attention(query, use_split=True)

diff = torch.abs(output_simple - output_split).mean()
print(f"Simple output shape: {output_simple.shape}")
print(f"Split  output shape: {output_split.shape}")
print(f"Mean absolute difference (simple vs split): {diff.item():.6f}")


In [ ]:
# Test 2: update loop — simple and split should track each other
for step in range(10):
    new_k = torch.randn(1, 4096)
    new_v = torch.randn(1, 4096)
    cache.update(new_k, new_v)
    out_s  = cache.compute_attention(query, use_split=False)
    out_sp = cache.compute_attention(query, use_split=True)
    delta = (out_s - out_sp).abs().max().item()
    print(f"Step {step:2d}: max |simple - split| = {delta:.6f}")


In [ ]:
# Test 3: memory stats API
cache2 = KIVICache()
cache2.prefill(keys, values)
mem_stats = cache2.get_memory_stats()
print(f"Theoretical KV memory: {mem_stats["total_mb"]:.2f} MB")


## Section 3: Memory Profiling

Measure theoretical KV-cache sizes and empirical GPU peak memory
for a short prompt and a long prompt.


### 3a. Short Prompt

In [ ]:
SHORT_PROMPT = "The capital of France is"
MAX_NEW_TOKENS = 40

baseline_mem_short = measure_baseline_memory(
    baseline_model, tokenizer, SHORT_PROMPT, max_new_tokens=MAX_NEW_TOKENS
)
print(f"[Baseline] GPU peak:        {baseline_mem_short["gpu_peak_mb"]:.2f} MB")
print(f"[Baseline] KV theoretical:  {baseline_mem_short["kv_theoretical_mb"]:.2f} MB")
print(f"[Baseline] Output:          {baseline_mem_short["output"]}")


In [ ]:
reset_all_kivi_caches(kivi_model)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

inputs = tokenizer(SHORT_PROMPT, return_tensors="pt").to(kivi_model.device)
with torch.no_grad():
    _ = kivi_model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        use_cache=True, pad_token_id=tokenizer.eos_token_id,
    )

kivi_gpu_peak_short = torch.cuda.max_memory_allocated() / (1024 ** 2)
kivi_stats_short    = get_kivi_memory_stats(kivi_model)

print(f"[KIVI] GPU peak:        {kivi_gpu_peak_short:.2f} MB")
print(f"[KIVI] KV theoretical:  {kivi_stats_short["total_mb"]:.2f} MB")
print_kivi_memory_stats(kivi_stats_short)


### 3b. Long Prompt

In [ ]:
LONG_PROMPT = (
    "Antibiotics are a type of medication used to treat bacterial infections. "
    "They work by either killing the bacteria or preventing them from reproducing, "
    "allowing the body's immune system to fight off the infection. "
    "Antibiotics are usually taken orally in the form of pills, capsules, or liquid "
    "solutions, or sometimes administered intravenously. They are not effective "
    "against viral infections, and using them inappropriately can lead to antibiotic "
    "resistance. Explain the above in one sentence:"
)

baseline_text_long, baseline_pkv_long = generate_baseline_with_output_and_kv(
    baseline_model, tokenizer, LONG_PROMPT, max_new_tokens=MAX_NEW_TOKENS
)
baseline_kv_long = get_baseline_kv_memory_stats(baseline_pkv_long)

print(f"[Baseline] KV total: {baseline_kv_long["total_mb"]:.2f} MB")
print(f"[Baseline] Output: {baseline_text_long}")


In [ ]:
reset_all_kivi_caches(kivi_model)
inputs = tokenizer(LONG_PROMPT, return_tensors="pt").to(kivi_model.device)
prompt_len = inputs["input_ids"].shape[-1]
with torch.no_grad():
    out_long = kivi_model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        use_cache=True, pad_token_id=tokenizer.eos_token_id,
    )
kivi_text_long  = tokenizer.decode(out_long[0][prompt_len:], skip_special_tokens=True)
kivi_kv_long    = get_kivi_memory_stats(kivi_model)

print(f"[KIVI] KV total: {kivi_kv_long["total_mb"]:.2f} MB")
print(f"[KIVI] Output: {kivi_text_long}")

reduction = baseline_kv_long["total_bytes"] / kivi_kv_long["total_bytes"]
match = token_level_match_rate(baseline_text_long, kivi_text_long, tokenizer)
print(f"Memory reduction: {reduction:.2f}x")
print(f"Token match rate: {match["token_match_rate"]:.4f}")


## Section 4: CNN/DailyMail Summarization Benchmark

Evaluates summarization quality on 100 CNN/DailyMail validation examples using
ROUGE-L, BERTScore, token match rate, sentence compliance, and entity hallucination rate.


In [ ]:
CNN_DM_SAMPLES     = 100
MAX_NEW_TOKENS_CNN = 256

cnn_dm_dataset = cnn_dm_bench.load_cnn_dm(split="validation", num_samples=CNN_DM_SAMPLES)
articles_cnn   = [s["article"] for s in cnn_dm_dataset]
print(f"Loaded {len(cnn_dm_dataset)} CNN/DM examples.")


In [ ]:
# Baseline generation pass — run first, then delete model to free GPU memory
del kivi_model
torch.cuda.empty_cache()

baseline_model = LlamaForCausalLM.from_pretrained(
    MODEL_7B, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True
)

baseline_outputs_cnn, prompts_cnn, references_cnn, baseline_mem_cnn = (
    cnn_dm_bench.generate_all_summaries(
        baseline_model, tokenizer, cnn_dm_dataset,
        label="BASELINE", max_new_tokens=MAX_NEW_TOKENS_CNN,
    )
)

del baseline_model
torch.cuda.empty_cache()
torch.cuda.synchronize()


In [ ]:
# KIVI generation pass
kivi_model = LlamaForCausalLM.from_pretrained(
    MODEL_7B, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True
)
kivi_model = replace_llama_attention_with_kivi(kivi_model)

kivi_outputs_cnn, _, _, kivi_mem_cnn = (
    cnn_dm_bench.generate_all_summaries(
        kivi_model, tokenizer, cnn_dm_dataset,
        label="KIVI", max_new_tokens=MAX_NEW_TOKENS_CNN,
    )
)

del kivi_model
torch.cuda.empty_cache()
torch.cuda.synchronize()


In [ ]:
# Evaluation (CPU only — no GPU needed)
examples_cnn, results_cnn = cnn_dm_bench.evaluate_on_cnn_dm_from_texts(
    articles=articles_cnn,
    references=references_cnn,
    prompts=prompts_cnn,
    baseline_outputs=baseline_outputs_cnn,
    kivi_outputs=kivi_outputs_cnn,
    tokenizer=tokenizer,
)

memory_results_cnn = {"baseline": baseline_mem_cnn, "kivi": kivi_mem_cnn}

table_cnn = cnn_dm_bench.build_results_table(results_cnn)
cnn_dm_bench.print_results_table(table_cnn)


In [ ]:
print_per_prompt_kv(memory_results_cnn)
print_aggregate_kv(memory_results_cnn)


In [ ]:
cnn_dm_bench.save_cnn_results(
    examples=examples_cnn,
    results=results_cnn,
    memory_results=memory_results_cnn,
    out_dir="results/cnn_dm_kivi_eval",
)


## Section 5: CoQA Conversational QA Benchmark

Evaluates conversational QA on 200 CoQA examples (LLaMA-2 7B)
using CoQA F1 (raw + robust), ROUGE-L, BERTScore, and token match rate.


In [ ]:
COQA_SAMPLES = 200

coqa_data = coqa_bench.load_coqa(n=COQA_SAMPLES, split="validation")
print(f"Loaded {len(coqa_data)} CoQA examples.")


In [ ]:
# Baseline generation pass
baseline_model = LlamaForCausalLM.from_pretrained(
    MODEL_7B, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True
)

baseline_outputs_coqa, ground_truths_coqa, baseline_mem_coqa = (
    coqa_bench.run_model_and_collect_outputs_with_kv(
        baseline_model, tokenizer, coqa_data, label="BASELINE", debug_n=3,
    )
)

del baseline_model
torch.cuda.empty_cache()
torch.cuda.synchronize()


In [ ]:
# KIVI generation pass
kivi_model = LlamaForCausalLM.from_pretrained(
    MODEL_7B, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True
)
kivi_model = replace_llama_attention_with_kivi(kivi_model)

kivi_outputs_coqa, _, kivi_mem_coqa = (
    coqa_bench.run_model_and_collect_outputs_with_kv(
        kivi_model, tokenizer, coqa_data, label="KIVI", debug_n=3,
    )
)

del kivi_model
torch.cuda.empty_cache()
torch.cuda.synchronize()


In [ ]:
# Metrics (CPU)
results_coqa = coqa_bench.evaluate_all_metrics(
    baseline_outputs=baseline_outputs_coqa,
    kivi_outputs=kivi_outputs_coqa,
    ground_truths=ground_truths_coqa,
    tokenizer=tokenizer,
    device="cpu",
)

aggregate_metrics_coqa = results_coqa["aggregate"]
examples_coqa = [
    {
        "prompt": coqa_bench.build_coqa_prompt(
            coqa_data[i]["context"], coqa_data[i]["question"]
        ),
        "ground_truth":    ground_truths_coqa[i],
        "baseline_output": baseline_outputs_coqa[i],
        "kivi_output":     kivi_outputs_coqa[i],
        "scores":          results_coqa["per_example"][i],
    }
    for i in range(len(ground_truths_coqa))
]

memory_results_coqa = {"baseline": baseline_mem_coqa, "kivi": kivi_mem_coqa}

table_coqa = coqa_bench.build_coqa_aggregate_table(aggregate_metrics_coqa)
coqa_bench.print_coqa_aggregate_table(table_coqa)


In [ ]:
print_all_memory_results(memory_results_coqa)


In [ ]:
coqa_bench.save_coqa_results(
    examples=examples_coqa,
    results=aggregate_metrics_coqa,
    memory_results=memory_results_coqa,
    out_dir="results/coqa_kivi_eval",
    n_samples=COQA_SAMPLES,
)


## Section 6: GSM8K Math Reasoning Benchmark (LLaMA-2 13B)

Evaluates math reasoning on 50 GSM8K test examples using LLaMA-2 13B.
Metrics: exact match (EM), consistency rate, token match rate.


In [ ]:
GSM8K_SAMPLES        = 50
MAX_NEW_TOKENS_GSM8K = 256

tokenizer_13b = AutoTokenizer.from_pretrained(MODEL_13B)
gsm8k_dataset = gsm8k_bench.load_gsm8k(split="test", num_samples=GSM8K_SAMPLES)
print(f"Loaded {len(gsm8k_dataset)} GSM8K test examples.")


In [ ]:
# Baseline generation pass (13B)
baseline_13b = LlamaForCausalLM.from_pretrained(
    MODEL_13B, torch_dtype=torch.float16, device_map="auto"
)

baseline_outputs_gsm8k, prompts_gsm8k, gt_answers_gsm8k, baseline_mem_gsm8k = (
    gsm8k_bench.generate_all_gsm8k_answers(
        baseline_13b, tokenizer_13b, gsm8k_dataset,
        label="BASELINE", max_new_tokens=MAX_NEW_TOKENS_GSM8K,
    )
)

del baseline_13b
torch.cuda.empty_cache()


In [ ]:
# KIVI generation pass (13B)
kivi_13b = LlamaForCausalLM.from_pretrained(
    MODEL_13B, torch_dtype=torch.float16, device_map="auto"
)
kivi_13b = replace_llama_attention_with_kivi(kivi_13b)

kivi_outputs_gsm8k, _, _, kivi_mem_gsm8k = (
    gsm8k_bench.generate_all_gsm8k_answers(
        kivi_13b, tokenizer_13b, gsm8k_dataset,
        label="KIVI", max_new_tokens=MAX_NEW_TOKENS_GSM8K,
    )
)

del kivi_13b
torch.cuda.empty_cache()


In [ ]:
# Evaluation (exact match from pre-generated texts)
examples_gsm8k, aggregate_gsm8k = gsm8k_bench.evaluate_on_gsm8k_from_texts(
    prompts=prompts_gsm8k,
    gt_answers=gt_answers_gsm8k,
    baseline_outputs=baseline_outputs_gsm8k,
    kivi_outputs=kivi_outputs_gsm8k,
    tokenizer=tokenizer_13b,
)

memory_results_gsm8k = {"baseline": baseline_mem_gsm8k, "kivi": kivi_mem_gsm8k}

table_gsm8k = gsm8k_bench.build_gsm8k_aggregate_table(aggregate_gsm8k)
gsm8k_bench.print_gsm8k_aggregate_table(table_gsm8k)


In [ ]:
print_all_memory_results(memory_results_gsm8k)


In [ ]:
gsm8k_bench.save_gsm8k_results_json(
    per_sample=examples_gsm8k,
    aggregate=aggregate_gsm8k,
    memory_results=memory_results_gsm8k,
    out_dir="results/gsm8k_13b_kivi_eval",
    prefix="gsm8k",
)
